In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

# 📍 Chemin vers le CSV consolidé
CSV_FILE = "C:/Users/julie/Data Eng/Processed/deliveries.csv"

# 🐘 Paramètres de connexion PostgreSQL
DB_USER = "postgres"
DB_PASS = "postgres"
DB_HOST = "127.0.0.1"  
DB_PORT = "5432"
DB_NAME = "deliveries_db"

# URL SQLAlchemy
DB_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

def load_csv_to_postgres():
    if not os.path.exists(CSV_FILE):
        print(f"❌ Fichier introuvable : {CSV_FILE}")
        return

    try:
        print("📥 Lecture du fichier CSV...")
        df = pd.read_csv(CSV_FILE, parse_dates=["start_time", "end_time"])

        expected_columns = {
            "delivery_id", "client_id", "driver_id", "start_time", "end_time",
            "status", "duration_minutes", "failure_reason", "event_count"
        }

        if not expected_columns.issubset(df.columns):
            print("❌ Le fichier CSV ne contient pas toutes les colonnes attendues.")
            print("Colonnes trouvées :", df.columns.tolist())
            return

        print("🔗 Connexion à PostgreSQL...")
        engine = create_engine(DB_URL)

        with engine.begin() as conn:  # ✅ begin = commit automatique si tout se passe bien
            print("🛠️ Création de la table si elle n'existe pas...")
            conn.execute(text("""
                CREATE TABLE IF NOT EXISTS deliveries (
                    delivery_id TEXT PRIMARY KEY,
                    client_id TEXT,
                    driver_id TEXT,
                    start_time TIMESTAMP,
                    end_time TIMESTAMP,
                    status TEXT,
                    duration_minutes INT,
                    failure_reason TEXT,
                    event_count INT
                );
            """))

            print("🧼 Suppression des anciennes données...")
            conn.execute(text("DELETE FROM deliveries;"))

        # 🚀 Insertion séparée (hors transaction) pour éviter les locks
        print("⬆️ Insertion des données (par chunk)...")
        df.to_sql("deliveries", con=engine, if_exists="append", index=False, chunksize=500)

        print("✅ Données chargées avec succès !")

    except SQLAlchemyError as e:
        print("❌ Erreur SQLAlchemy :", e)
    except Exception as e:
        print("❌ Erreur inattendue :", e)

if __name__ == "__main__":
    load_csv_to_postgres()


📥 Lecture du fichier CSV...
🔗 Connexion à PostgreSQL...
🛠️ Création de la table si elle n'existe pas...
🧼 Suppression des anciennes données...
⬆️ Insertion des données (par chunk)...
✅ Données chargées avec succès !
